# [실습 12] 트레이싱 — 실행 발자취(스팬) 기록하기

> **연계**: 제5부 12장(Observability) · **환경**: Google Colab · Python

**학습 목표**
- 데코레이터로 각 단계(**스팬**)의 입력·출력·지연·비용을 자동 기록한다(12-1).
- 실행 후 **트레이스**를 재구성해 병목을 찾는다.

## 1. 스팬 기록 데코레이터

In [ ]:
import time, functools

TRACE = []  # 트레이스 = 스팬들의 모음

def span(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        t0 = time.time()
        result = fn(*args, **kwargs)
        TRACE.append({
            "span": fn.__name__,
            "input": str(args)[:40],
            "output": str(result)[:40],
            "latency_ms": round((time.time() - t0) * 1000, 1),
        })
        return result
    return wrapper

## 2. 에이전트 단계에 스팬을 붙여 실행

In [ ]:
@span
def plan(goal):
    time.sleep(0.05); return f"{goal} 계획"

@span
def search(q):
    time.sleep(0.30); return "검색결과"   # 느린 단계(병목)

@span
def generate(ctx):
    time.sleep(0.08); return "최종답변"

def agent(goal):
    p = plan(goal); s = search(p); return generate(s)

agent("환율 알려줘")

## 3. 트레이스 재구성 → 병목 찾기

In [ ]:
print("=== 트레이스 ===")
for s in TRACE:
    print(f"  [{s['span']:9}] {s['latency_ms']:>6} ms | in={s['input']} out={s['output']}")
slowest = max(TRACE, key=lambda s: s['latency_ms'])
print("\n병목 스팬:", slowest['span'], slowest['latency_ms'], "ms")

## 4. 정리
- 각 단계를 **스팬**으로 기록해 **트레이스**를 재구성했다(12-1).
- 지연을 스팬별로 분해해 **병목(search)** 을 찾았다.
- **더 해보기**: 각 스팬에 토큰 수·비용 필드를 추가해 비용 병목도 찾아보세요.